In [ ]:
df = (spark.readStream
      .format("kafka")
      .option("kafka.bootstrap.servers", "kafka-3ad4f62-project-746a.c.***************")
      .option("subscribe", "ride_requests")
      .option("startingOffsets", "latest")
      .option("kafka.security.protocol", "SSL")
      .option("kafka.ssl.keystore.location", "/Volumes/workspace/default/volumekafka/kafka-keystore.jks")
      .option("kafka.ssl.keystore.password", "changeit")
      .option("kafka.ssl.truststore.location", "/Volumes/workspace/default/volumekafka/kafka-truststore.jks")
      .option("kafka.ssl.truststore.password", "changeit")
      .load())


In [ ]:
df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)") \
  .writeStream \
  .format("console") \
  .outputMode("append") \
  .option("checkpointLocation", "/Volumes/workspace/default/streaming_checkpoints/ride_requests") \
  .trigger(availableNow=True) \
  .start()

In [ ]:
from pyspark.sql.functions import col, from_json
from pyspark.sql.types import StructType, StructField, StringType

# Definir el esquema del JSON
schema = StructType([
    StructField("ride_id", StringType(), True),
    StructField("user_id", StringType(), True),
    StructField("pickup", StringType(), True),
    StructField("destination", StringType(), True)
])

In [ ]:
parsed_df = df.select(from_json(col("value").cast("string"), schema).alias("data")).select("data.*")

In [ ]:
query = (parsed_df.writeStream
         .format("json")  # guarda como archivos JSON
         .option("path", "/Volumes/workspace/default/volume1")  # tu volumen destino
         .option("checkpointLocation", "/Volumes/workspace/default/streaming_checkpoints/ride_requests")
         .outputMode("append")
         .trigger(availableNow=True)
         #.trigger(processingTime="10 seconds")
         .start())

In [ ]:
df_saved = spark.read.json("/Volumes/workspace/default/volume1/*.json")
display(df_saved)

In [ ]:
%sql

CREATE TABLE IF NOT EXISTS workspace.default.ride_request_kafka (
  ride_id STRING,
  user_id STRING,
  pickup STRING,
  destination STRING
)
USING DELTA;

In [ ]:
query = (parsed_df.writeStream
         .format("delta")
         .option("checkpointLocation", "/Volumes/workspace/default/streaming_checkpoints/ride_requests_kafka")
         .outputMode("append")
         .trigger(availableNow=True)
         .table("workspace.default.ride_request_kafka"))

In [ ]:
%sql

SELECT * FROM workspace.default.ride_request_kafka